# MASC — Inference & Đánh giá Metrics (FID, IS, Precision, Recall)

Notebook chuyên dụng để **Sinh ảnh** và **Đo các chỉ số chất lượng** từ mô hình đã huấn luyện xong (`ckpt_final.pt` + `masc_mapping_k4096.npz`).

> ⚡ **Không cần train lại, không cần tải 7GB dữ liệu ImageNet!**
> ⏱️ **Thời gian setup:** ~1-2 phút.

In [ ]:
# ============================================================
# CELL 1 — Setup môi trường (Clone repos & cài đặt thư viện)
# ============================================================
import os, sys, subprocess

WORK_DIR  = '/kaggle/working'
MASC_DIR  = f'{WORK_DIR}/MASC'
LLAMA_DIR = f'{WORK_DIR}/LlamaGen'

def run(cmd):
    print(f'>>> {cmd}')
    return subprocess.run(cmd, shell=True).returncode

# Clone / Cập nhật MASC repo
if not os.path.exists(MASC_DIR):
    run(f'git clone https://github.com/nta2112/masc-upgrade-ann {MASC_DIR}')
else:
    run(f'git -C {MASC_DIR} fetch origin && git -C {MASC_DIR} reset --hard origin/main')

# Clone LlamaGen repo
if not os.path.exists(LLAMA_DIR):
    run(f'git clone https://github.com/FoundationVision/LlamaGen {LLAMA_DIR}')

for p in [MASC_DIR, LLAMA_DIR]:
    if p not in sys.path: sys.path.insert(0, p)

# Cài các thư viện cần cho sampling & evaluation
run('pip install -q pyyaml huggingface_hub scipy tqdm')
print('\n✅ Cell 1 done — Môi trường đã sẵn sàng!')


In [ ]:
# ============================================================
# CELL 2 — Tải VQ-VAE Decoder weights (vq_ds16_c2i.pt ~288 MB)
# (Dùng để giải mã các token ảnh thành điểm ảnh RGB)
# ============================================================
from huggingface_hub import hf_hub_download
import os

WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

vq_path = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
if not os.path.exists(vq_path):
    print('Dang tai vq_ds16_c2i.pt (~288 MB)...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename='vq_ds16_c2i.pt', local_dir=WEIGHTS_DIR)
    print('✅ Da tai xong VQ-VAE!')
else:
    print(f'✅ vq_ds16_c2i.pt da co san ({os.path.getsize(vq_path)/1e6:.0f} MB)')

print('Cell 2 done')


In [ ]:
# ============================================================
# CELL 3 — Tự động tìm Checkpoint ckpt_final.pt và MASC Mapping
# ============================================================
import os, glob

# 1. Tìm checkpoint ckpt_final.pt
ckpt_candidates = (
    glob.glob('/kaggle/working/**/ckpt_final.pt', recursive=True) +
    glob.glob('/kaggle/input/**/ckpt_final.pt', recursive=True) +
    glob.glob('/kaggle/working/**/ckpt_*.pt', recursive=True) +
    glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)
)
if not ckpt_candidates:
    raise FileNotFoundError(
        '❌ Không tìm thấy ckpt_final.pt! '\
        'Hãy upload ckpt_final.pt lên Kaggle Dataset (bấm Add Input -> Upload Dataset)!'
    )
CKPT_PATH = ckpt_candidates[0]
print(f'✅ Tìm thấy Checkpoint: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)')

# 2. Tìm file mapping masc_mapping_k4096.npz
map_candidates = (
    glob.glob('/kaggle/working/**/masc_mapping*.npz', recursive=True) +
    glob.glob('/kaggle/input/**/masc_mapping*.npz', recursive=True)
)
if not map_candidates:
    raise FileNotFoundError(
        '❌ Không tìm thấy masc_mapping_k4096.npz! '\
        'Hãy upload masc_mapping_k4096.npz lên Kaggle Dataset!'
    )
MAP_PATH = map_candidates[0]
print(f'✅ Tìm thấy MASC Mapping: {MAP_PATH}')
print('Cell 3 done')


In [ ]:
# ============================================================
# CELL 4 — Khởi tạo & Load mô hình vào GPU
# ============================================================
import torch
from masc import load_mapping
from tokenizer.tokenizer_image.vq_model import VQ_models
from autoregressive.models.gpt import GPT_models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

# 1. Load MASC mapping
mp = load_mapping(MAP_PATH)
print(f'MASC Mapping: N={mp.n} fine tokens -> k={mp.k} coarse clusters')

# 2. Load VQ-VAE Decoder
vq_model = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
vq_model.to(device).eval()
vq_state = torch.load('/kaggle/working/weights/vq_ds16_c2i.pt', map_location='cpu')
vq_model.load_state_dict(vq_state.get('model', vq_state.get('ema', vq_state)))
print('VQ-VAE Tokenizer Decoder đã load thành công!')

# 3. Load LlamaGen-L với từ vựng MASC k=4096
latent_size = 256 // 16  # 16x16 = 256 tokens
gpt_model = GPT_models['GPT-L'](
    vocab_size=mp.k,
    block_size=latent_size ** 2,
    num_classes=1000,
    cls_token_num=1,
    model_type='c2i',
).to(device=device, dtype=torch.float16)

ckpt = torch.load(CKPT_PATH, map_location='cpu')
gpt_model.load_state_dict(ckpt.get('model', ckpt), strict=False)
gpt_model.eval()
print('LlamaGen-L MASC Generator đã load thành công!')
print('Cell 4 done — Sẵn sàng sinh ảnh & đánh giá metrics!')


In [ ]:
# ============================================================
# CELL 5 — DEMO SINH ẢNH (Image Generation)
# ============================================================
import time
from torchvision.utils import save_image
from IPython.display import Image, display
from autoregressive.models.generate import generate
from masc import random_sample_decode

@torch.no_grad()
def generate_images(class_labels, cfg_scale=2.5, temperature=1.0, top_k=2000, seed=42):
    torch.manual_seed(seed)
    c_indices = torch.tensor(class_labels, device=device)
    B = len(class_labels)
    qzshape = [B, 8, latent_size, latent_size]

    print(f'Đang sinh {B} ảnh (cfg={cfg_scale}, temp={temperature})...')
    t0 = time.time()
    # 1. AR coarse token sampling (k=4096)
    coarse = generate(gpt_model, c_indices, latent_size**2,
                      cfg_scale=cfg_scale, temperature=temperature, top_k=top_k, sample_logits=True)
    
    # 2. MASC decode: coarse clusters -> fine tokens
    fine = random_sample_decode(coarse.detach().cpu().numpy(), mp)
    fine_t = torch.as_tensor(fine, device=device)
    
    # 3. VQ-VAE decode: tokens -> ảnh điểm ảnh RGB [-1, 1]
    images = vq_model.decode_code(fine_t, qzshape)
    print(f'Hoàn tất trong {time.time()-t0:.2f} giây!')

    out_path = '/kaggle/working/demo_samples.png'
    save_image(images, out_path, nrow=min(4, B), normalize=True, value_range=(-1, 1))
    display(Image(out_path))
    return out_path

# Gợi ý một số Class ImageNet:
# 207: Chó Golden | 985: Hoa cúc | 387: Voi | 88: Vẹt | 1: Cá vàng | 417: Khinh khí cầu
demo_classes = [207, 985, 387, 88, 1, 417, 974, 279]
generate_images(demo_classes, cfg_scale=2.5, seed=100)


In [ ]:
# ============================================================
# CELL 6 — ĐÁNH GIÁ METRICS: FID, Inception Score, Precision, Recall
# ============================================================
import os, sys, time, subprocess
import numpy as np, torch
from tqdm import tqdm
from autoregressive.models.generate import generate
from masc import random_sample_decode

# --- Cấu hình đánh giá ---
# NUM_SAMPLES: 1000 để chạy nhanh (~10-12 phút), hoặc 5000 / 50000 theo chuẩn
NUM_SAMPLES = 1000
BATCH_SIZE  = 16
CFG_SCALE   = 2.5
SAMPLE_NPZ  = f'/kaggle/working/samples_{NUM_SAMPLES}.npz'
REF_NPZ     = '/kaggle/working/VIRTUAL_imagenet256_labeled.npz'

# 1. Tải Reference Statistics của OpenAI ImageNet-256 (nếu chưa có)
if not os.path.exists(REF_NPZ):
    print('Đang tải file thống kê ImageNet chuẩn (VIRTUAL_imagenet256_labeled.npz)...')
    url = 'https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz'
    subprocess.run(f'curl -L -o {REF_NPZ} {url}', shell=True, check=True)
    print('✅ Đã tải xong Reference Batch!')

# 2. Sinh N ảnh mẫu lưu vào file .npz
if not os.path.exists(SAMPLE_NPZ):
    print(f'\nBắt đầu sinh {NUM_SAMPLES} ảnh mẫu (CFG={CFG_SCALE})...')
    all_images = []
    num_batches = (NUM_SAMPLES + BATCH_SIZE - 1) // BATCH_SIZE
    rng = np.random.default_rng(42)
    classes = rng.choice(1000, size=NUM_SAMPLES, replace=True)

    with torch.no_grad():
        for b in tqdm(range(num_batches), desc='Sinh ảnh đánh giá'):
            b_classes = classes[b * BATCH_SIZE : min((b + 1) * BATCH_SIZE, NUM_SAMPLES)]
            cur_bs = len(b_classes)
            c_indices = torch.tensor(b_classes, device=device)
            
            coarse = generate(gpt_model, c_indices, latent_size**2,
                              cfg_scale=CFG_SCALE, temperature=1.0, top_k=2000, sample_logits=True)
            fine = random_sample_decode(coarse.detach().cpu().numpy(), mp)
            fine_t = torch.as_tensor(fine, device=device)
            
            imgs = vq_model.decode_code(fine_t, [cur_bs, 8, latent_size, latent_size])
            # Đưa về dạng chuẩn [B, 256, 256, 3] định dạng uint8 trong [0, 255]
            imgs = (imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0
            imgs_np = imgs.permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            all_images.append(imgs_np)

    all_images = np.concatenate(all_images, axis=0)[:NUM_SAMPLES]
    np.savez_compressed(SAMPLE_NPZ, arr_0=all_images)
    print(f'✅ Đã lưu {len(all_images)} ảnh vào: {SAMPLE_NPZ}')
else:
    print(f'File {SAMPLE_NPZ} đã có sẵn, bỏ qua bước sinh ảnh!')

# 3. Chạy bộ đo LlamaGen chính thức
print('\n' + '=' * 60)
print('TIẾN HÀNH TÍNH TOÁN METRICS (FID, Inception Score, Precision, Recall)...')
print('=' * 60)
eval_cmd = f'python /kaggle/working/LlamaGen/evaluations/c2i/evaluator.py {REF_NPZ} {SAMPLE_NPZ}'
proc = subprocess.Popen(eval_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n🏆 HOÀN TẤT ĐÁNH GIÁ METRICS!')
